# Практическое занятие 2. Извлечение данных из заявок на аренду жилья

Решение: LangChain + GigaChat извлекают количество проживающих из
неструктурированных текстов заявок.

Ключ доступа хранится в файле `.env` в переменной `GIGA_KEY` и не попадает
в репозиторий.

## 1. Подключение к GigaChat

In [1]:
import os

from dotenv import load_dotenv
from langchain_gigachat.chat_models import GigaChat

load_dotenv()
giga_key = os.getenv("GIGA_KEY")

if not giga_key:
    raise ValueError("Ключ GIGA_KEY не найден в .env")

llm = GigaChat(
    credentials=giga_key,
    model="GigaChat-2",
    verify_ssl_certs=False,
    temperature=0.2,
    max_tokens=1000,
)

print(llm.invoke("Привет! Как дела?").content)

Привет! Всё отлично, спасибо. Готов обсудить что-нибудь интересное или помочь разобраться с задачей. Чем займёмся?


## 2. Базовый промпт и цепочка LangChain

Промпт формулирует задачу, модель её выполняет, парсер `StrOutputParser`
возвращает результат как строку.

In [2]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

basic_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Проанализируй следующий текст заявки на аренду жилья и извлеки количество человек, которые будут проживать.
Текст заявки: {text}
Верни только число (целое число), соответствующее количеству проживающих.
Если количество не указано явно, постарайся определить его по контексту.
Количество человек:""",
)

chain = basic_prompt | llm | StrOutputParser()

test_texts = [
    "Ищу квартиру для семьи из четырех человек",
    "Нужна студия для одного",
    "Семья с двумя детьми ищет жилье",
]

for text in test_texts:
    print(f"Текст: {text}")
    print(f"Результат: {chain.invoke({'text': text})}")
    print("---")

Текст: Ищу квартиру для семьи из четырех человек


Результат: 4
---
Текст: Нужна студия для одного


Результат: 1
---
Текст: Семья с двумя детьми ищет жилье
Результат: 4
---


## 3. Обработка моих 15 заявок (вариант 17)

Заявки берутся из файла `rental_17.csv`. Ответы модели собираются в столбец
`result`, итоговая таблица сохраняется в `rental_with_results.csv`.

In [3]:
import pandas as pd

df = pd.read_csv("rental_17.csv", sep=";")

results = []
for _, row in df.iterrows():
    text = row["text"]
    try:
        results.append(chain.invoke({"text": text}))
    except Exception as error:
        results.append(f"ERROR: {error}")

df["result"] = results
df.to_csv("rental_with_results.csv", index=False, encoding="utf-8-sig")
df[["amount", "result"]]

,amount,result
0,6,6
1,2,2
2,4,4
3,3,3
4,3,3
5,3,3
6,3,3
7,4,4
8,4,4
9,2,2


## 4. Оценка точности базовой цепочки

Сравниваем ответы модели из столбца `result` с эталонным `amount`.

In [4]:
import pandas as pd

# Типы данных: столбец result хранит строки (object)
print(df.dtypes)

# Преобразуем ответы модели в числа; нечисловые ответы станут NaN
df["result_num"] = pd.to_numeric(df["result"], errors="coerce")
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

correct = (df["result_num"] == df["amount"]).sum()
total = len(df)
errors = total - correct

print("Ошибок:", errors)
print("Точность: {:.1%}".format(correct / total))

# Строки, где ответ модели не совпал с эталоном
df.loc[df["result_num"] != df["amount"], ["amount", "result", "result_num"]]

amount    int64
text        str
result      str
dtype: object
Ошибок: 1
Точность: 93.3%


,amount,result,result_num
13,1,Не удалось извлечь конкретное количество челов...,NaN


## 5. Сравнение техник промптирования

Проверяем на тех же 15 заявках три техники: детализированные инструкции,
few-shot и Chain-of-Thought.

In [5]:
import re

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

detailed_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Ты — эксперт по анализу текстов заявок на аренду жилья. Твоя задача — извлечь точное количество людей, которые будут проживать в арендуемом жилье.
ИНСТРУКЦИИ:
1. Внимательно прочитай текст заявки
2. Найди все упоминания количества людей (прямые и косвенные)
3. Учти следующие правила:
- "семья с одним ребенком" = 3 человека (родители + ребенок)
- "семья с двумя детьми" = 4 человека (родители + дети)
- "пара", "двое", "молодая семья" = 2 человека
- "один", "студент", "индивидуально" = 1 человек
4. Если количество указано числом (например, "троих", "четверых"), используй это число
5. Верни ТОЛЬКО целое число без дополнительных символов или пояснений

Текст заявки: {text}
Количество проживающих:""",
)

few_shot_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Извлеки количество людей из текста заявки на аренду жилья. Изучи примеры:
ПРИМЕРЫ:
Заявка: "Семья из трех человек ищет двухкомнатную квартиру"
Количество: 3
Заявка: "Молодая пара снимет студию в центре города"
Количество: 2
Заявка: "Ищу однокомнатную квартиру для себя"
Количество: 1
Заявка: "Семья с одним ребенком рассматривает варианты жилья"
Количество: 3
Заявка: "Четверо студентов ищут четырехкомнатную квартиру"
Количество: 4

Теперь проанализируй эту заявку:
Заявка: {text}
Количество:""",
)

cot_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Проанализируй заявку на аренду жилья и определи количество проживающих, следуя пошаговому алгоритму:
Заявка: {text}
ШАГ 1: Найди все упоминания людей в тексте
ШАГ 2: Определи значение каждого упоминания
- "семья с N детьми" = 2 родителя + N детей
- "пара" = 2 человека
- числительные переведи в цифры
ШАГ 3: Сложи общее количество
ШАГ 4: Верни итоговое число

Пошаговый анализ:
ШАГ 1:
ШАГ 2:
ШАГ 3:
ШАГ 4: Итоговое число:""",
)

techniques = {
    "Детализированный": detailed_prompt,
    "Few-shot": few_shot_prompt,
    "Chain-of-Thought": cot_prompt,
}


def to_int(value):
    if not isinstance(value, str):
        return None
    numbers = re.findall(r"-?\d+", value)
    return int(numbers[-1]) if numbers else None


for name, prompt in techniques.items():
    tech_chain = prompt | llm | StrOutputParser()
    predictions = []
    for _, row in df.iterrows():
        try:
            predictions.append(to_int(tech_chain.invoke({"text": row["text"]})))
        except Exception as error:
            print("ERROR:", error)
            predictions.append(None)
    correct_tech = sum(1 for value, amount in zip(predictions, df["amount"]) if value == amount)
    print(f"{name}: {correct_tech}/{len(df)} = {correct_tech / len(df):.1%}")

Детализированный: 15/15 = 100.0%


Few-shot: 15/15 = 100.0%


Chain-of-Thought: 12/15 = 80.0%


## 6. Структурированный JSON-вывод (задание № 2)

Расширяем решение: помимо количества проживающих извлекаем отдельно
число взрослых и детей, дату заезда, число ночей, цену за сутки и особые
пожелания. Используется `ChatPromptTemplate` из системного и
пользовательского сообщений, техника few-shot и `PydanticOutputParser`
для валидации JSON по схеме.

In [6]:
import json
from typing import Optional

import pandas as pd
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    SystemMessagePromptTemplate,
)
from pydantic import BaseModel, Field


class RentalRequest(BaseModel):
    count_adults: Optional[int] = Field(None, description="Число взрослых проживающих")
    count_children: Optional[int] = Field(None, description="Число детей проживающих")
    start_date: Optional[str] = Field(None, description="Дата заезда в формате ДД.ММ")
    nights: Optional[int] = Field(None, description="Число ночей проживания")
    price_per_day: Optional[int] = Field(None, description="Желаемая цена в сутки, рублей")
    remarks: Optional[str] = Field(None, description="Особые пожелания: животные, детская кроватка, парковка и т.п.")


parser = PydanticOutputParser(pydantic_object=RentalRequest)

system_prompt = SystemMessagePromptTemplate.from_template(
    """Ты — эксперт по анализу текстовых заявок на аренду жилья.
Извлекай общее количество проживающих и дополнительные поля строго по тексту заявки.
ПРАВИЛА:
- "семья с N детьми" = 2 взрослых + N детей
- при "плавающих" диапазонах дат бери самую раннюю дату заезда
- при "плавающих" диапазонах длительности бери максимальное число ночей
- при диапазоне цены бери максимум
- особые пожелания (животные, детская кроватка, парковка, кондиционер) записывай в remarks
- если поле в заявке не указано, верни null

Формат ответа:
{format_instructions}"""
)

# Техника few-shot: в пользовательском сообщении даём примеры перед самой заявкой
human_prompt = HumanMessagePromptTemplate.from_template(
    """ПРИМЕРЫ:
Заявка: "Семья с двумя детьми ищет квартиру с 5 июля на 10 дней, до 3000 за сутки"
{{"count_adults": 2, "count_children": 2, "start_date": "05.07", "nights": 10, "price_per_day": 3000, "remarks": null}}
Заявка: "Нужна студия для одного человека с собакой, с 1 по 8 августа, 2000-2500 руб/сутки"
{{"count_adults": 1, "count_children": 0, "start_date": "01.08", "nights": 7, "price_per_day": 2500, "remarks": "с собакой"}}

Заявка: {text}
Верни только JSON по схеме."""
)

chat_prompt = ChatPromptTemplate.from_messages([system_prompt, human_prompt]).partial(
    format_instructions=parser.get_format_instructions()
)

structured_chain = chat_prompt | llm | parser

predicted = []
for _, row in df.iterrows():
    try:
        parsed = structured_chain.invoke({"text": row["text"]})
        predicted.append(parsed.model_dump())
    except Exception as error:
        predicted.append({"error": f"{type(error).__name__}: {error}"})
    print(json.dumps(predicted[-1], ensure_ascii=False))

structured_df = pd.DataFrame(predicted)
structured_df

{"count_adults": 3, "count_children": 3, "start_date": "30.07", "nights": 9, "price_per_day": null, "remarks": null}


{"count_adults": 2, "count_children": 0, "start_date": "03.09", "nights": 7, "price_per_day": null, "remarks": null}


{"count_adults": 4, "count_children": null, "start_date": "31.08", "nights": 7, "price_per_day": null, "remarks": null}


{"count_adults": 2, "count_children": 1, "start_date": "08.07", "nights": 13, "price_per_day": null, "remarks": null}


{"count_adults": 2, "count_children": 1, "start_date": "17.08", "nights": 8, "price_per_day": null, "remarks": null}


{"count_adults": 3, "count_children": 0, "start_date": "08.07", "nights": 7, "price_per_day": null, "remarks": null}


... (6 outputs left out)

{"count_adults": 2, "count_children": 2, "start_date": "08.08", "nights": 13, "price_per_day": null, "remarks": null}


{"count_adults": null, "count_children": null, "start_date": null, "nights": null, "price_per_day": null, "remarks": null}


{"count_adults": 2, "count_children": 0, "start_date": "01.07", "nights": null, "price_per_day": null, "remarks": null}


,count_adults,count_children,start_date,nights,price_per_day,remarks,error
0,3.0,3.0,30.07,9.0,NaN,NaN,NaN
1,2.0,0.0,03.09,7.0,NaN,NaN,NaN
2,4.0,NaN,31.08,7.0,NaN,NaN,NaN
3,2.0,1.0,08.07,13.0,NaN,NaN,NaN
4,2.0,1.0,17.08,8.0,NaN,NaN,NaN
5,3.0,0.0,08.07,7.0,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,RemoteProtocolError: Server disconnected witho...
7,2.0,2.0,10.08,10.0,NaN,NaN,NaN
8,2.0,2.0,13.07,10.0,NaN,"душ, туалет, холодильник, кондиционер в номере...",NaN
9,1.0,1.0,06.09,10.0,NaN,NaN,NaN


## 7. Точность по дополнительным полям

Сравниваем извлечённые значения с ручной разметкой `rental_17_manual.csv`
по каждому полю и считаем среднюю точность.

In [7]:
import re

import pandas as pd

manual = pd.read_csv("rental_17_manual.csv", sep=";")


def norm_int(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def norm_date(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    match = re.search(r"(\d{1,2})[.\-/](\d{1,2})", str(value))
    if match:
        return f"{int(match.group(1)):02d}.{int(match.group(2)):02d}"
    return None


fields = {
    "count_adults": norm_int,
    "count_children": norm_int,
    "start_date": norm_date,
    "nights": norm_int,
    "price_per_day": norm_int,
}

total_correct = 0
total_rows = 0
manual_rows = manual.to_dict("records")
for field, normalizer in fields.items():
    correct = 0
    for predicted_row, manual_row in zip(predicted, manual_rows):
        expected = normalizer(manual_row[field])
        actual = normalizer(predicted_row.get(field))
        if expected == actual:
            correct += 1
        else:
            print("НЕ СОВПАЛО:", field, "| эталон:", expected, "| ответ:", actual)
    print(f"{field}: {correct}/{len(manual_rows)} = {correct / len(manual_rows):.1%}")
    total_correct += correct
    total_rows += len(manual_rows)

print("Средняя точность по полям:", f"{total_correct / total_rows:.1%}")

НЕ СОВПАЛО: count_adults | эталон: None | ответ: 3
НЕ СОВПАЛО: count_adults | эталон: 2 | ответ: None
НЕ СОВПАЛО: count_adults | эталон: 1 | ответ: None
count_adults: 12/15 = 80.0%
НЕ СОВПАЛО: count_children | эталон: 0 | ответ: None
НЕ СОВПАЛО: count_children | эталон: None | ответ: 0
НЕ СОВПАЛО: count_children | эталон: 1 | ответ: None
НЕ СОВПАЛО: count_children | эталон: 0 | ответ: None
count_children: 11/15 = 73.3%
НЕ СОВПАЛО: start_date | эталон: 17.08 | ответ: None
НЕ СОВПАЛО: start_date | эталон: None | ответ: 01.06
НЕ СОВПАЛО: start_date | эталон: None | ответ: 01.07
start_date: 12/15 = 80.0%
НЕ СОВПАЛО: nights | эталон: 10 | ответ: 9
НЕ СОВПАЛО: nights | эталон: None | ответ: 7
НЕ СОВПАЛО: nights | эталон: 10 | ответ: None
nights: 12/15 = 80.0%
price_per_day: 15/15 = 100.0%
Средняя точность по полям: 82.7%
